### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import numpy as np


In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.prompts import ChatPromptTemplate


In [3]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-3.5-turbo"
)


In [13]:
#  If your ChromaDB was created with persistence, you can connect to it like this

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectordb = Chroma(
    persist_directory="./chroma_rag_db",
    embedding_function=embeddings,
    collection_name="kcj-langchain-rag-chromadb"
)

retriever = vectordb.as_retriever()

In [14]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

 
rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000022FB97B1A30>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022FB5F30FB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022FB7464B60>, root_client

In [15]:
response = rag_chain_lcel.invoke("What is RAG?")
response

# response=rag_chain_lcel.invoke("What is Deep Learning")
# response

'RAG (Retrieval-Augmented Generation) is a technique that combines retrieval-based methods with generative models to enhance the quality and relevance of generated content. It retrieves relevant information from a knowledge base to inform the generation process, improving accuracy and context-awareness.'

In [16]:
retriever.invoke("What is Generative AI?")

[Document(id='d7cc7818-bac2-4749-847e-00d25ecff67d', metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpdp35w7pg\\doc_4.txt'}, page_content='Generative AI\n    Generative AI refers to a class of artificial intelligence algorithms that can generate new\n    content, such as text, images, music, or code, based on the data they have been trained on.\n    Prominent examples of generative AI include Generative Adversarial Networks (GANs) and transformer-based models\n    RAG (Retrieval-Augmented Generation) is a technique that combines retrieval-based methods with generative models to enhance\n    the quality and relevance of generated'),
 Document(id='d754ffef-4241-4183-ac8a-3e4bcfa79a6c', metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpi951hn87\\doc_3.txt'}, page_content='Artificial Intelligence \n    Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,\n    especially computer systems. These processes include learning (t

In [18]:
retriever.invoke("What is RAG?")

[Document(id='d7cc7818-bac2-4749-847e-00d25ecff67d', metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpdp35w7pg\\doc_4.txt'}, page_content='Generative AI\n    Generative AI refers to a class of artificial intelligence algorithms that can generate new\n    content, such as text, images, music, or code, based on the data they have been trained on.\n    Prominent examples of generative AI include Generative Adversarial Networks (GANs) and transformer-based models\n    RAG (Retrieval-Augmented Generation) is a technique that combines retrieval-based methods with generative models to enhance\n    the quality and relevance of generated'),
 Document(id='02217967-5f88-4909-b2b6-6b24cb403d7d', metadata={'source': 'C:\\Users\\harry\\AppData\\Local\\Temp\\tmpi951hn87\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(id='1ec1b99f-a22b-4335-8169-62f7d8a5b25d', metadata={'source': 'C:\

In [23]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [24]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What is RAG?")

Testing LCEL Chain:
Question: What is RAG?
--------------------------------------------------
Answer: RAG (Retrieval-Augmented Generation) is a technique that combines retrieval-based methods with generative models to enhance the quality and relevance of generated content. It retrieves relevant information from a knowledge base to inform the generation process, improving accuracy and context-awareness.

Source Documents:

--- Source 1 ---
Generative AI
    Generative AI refers to a class of artificial intelligence algorithms that can generate new
    content, such as text, images, music, or code, based on the data they have been traine...

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 4 ---
the quality and relevance of generated content. It retrieves relevant inf

In [25]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What is BlockChain?")

Testing LCEL Chain:
Question: What is BlockChain?
--------------------------------------------------
Answer: Based on the context provided, there is no information about BlockChain.

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....

--- Source 4 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....
